In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from helpers.pathlib import get_path_site_checkpoint
from tqdm import tqdm

from ata_pipeline1.helpers.enums import FieldNew, FieldSnowplow
from ata_pipeline1.preprocessors import (
    AddFieldEventParentId,
    AggregatePageActivities,
    SetRowIndex,
)
from ata_pipeline1.site import AFRO_LA, DALLAS_FREE_PRESS, OPEN_VALLEJO, THE_19TH

In [ ]:
CSV_PREFIX = "221103-221214"

In [ ]:
# Read data from CHECKPOINT 1
df_afla = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 1, AFRO_LA.name))
df_dfp = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 1, DALLAS_FREE_PRESS.name))
df_ov = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 1, OPEN_VALLEJO.name))
df_19 = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 1, THE_19TH.name))

In [ ]:
df_afla = SetRowIndex()(df_afla)
df_dfp = SetRowIndex()(df_dfp)
df_ov = SetRowIndex()(df_ov)
df_19 = SetRowIndex()(df_19)

In [ ]:
def find_parents_then_group(
    df: pd.DataFrame,
    ping_interval_noise_seconds: float,
    trivial_dwell_secs_cutoff: float = 1.0,
    trivial_users_dwell_sec_cutoff: float = 60.0,
):
    assert df[FieldSnowplow.DERIVED_TSTAMP].is_monotonic_increasing
    df = AddFieldEventParentId(ping_interval_noise_seconds=ping_interval_noise_seconds)(df, log_result=False)
    df_agg = AggregatePageActivities(
        agg_funcs={
            FieldSnowplow.DOMAIN_USERID: (FieldSnowplow.DOMAIN_USERID, "first"),
            FieldNew.DWELL_SECS: (FieldSnowplow.DERIVED_TSTAMP, lambda x: (x.max() - x.min()).total_seconds()),
        }
    )(df, log_result=False)

    num_parents = df[FieldNew.EVENT_PARENT_ID].nunique()
    num_events_trivial = (df_agg[FieldNew.DWELL_SECS] <= trivial_dwell_secs_cutoff).sum()
    perc_events_trivial = num_events_trivial / df_agg.shape[0] * 100
    mean_dwell_secs = df_agg[FieldNew.DWELL_SECS].mean()
    median_dwell_secs = df_agg[FieldNew.DWELL_SECS].median()
    num_users_trivial = (
        df_agg.groupby(FieldSnowplow.DOMAIN_USERID)[FieldNew.DWELL_SECS].sum() <= trivial_users_dwell_sec_cutoff
    ).sum()
    perc_users_trivial = num_users_trivial / df_agg[FieldSnowplow.DOMAIN_USERID].nunique() * 100

    return (
        ping_interval_noise_seconds,
        num_parents,
        num_events_trivial,
        perc_events_trivial,
        mean_dwell_secs,
        median_dwell_secs,
        num_users_trivial,
        perc_users_trivial,
    )

In [ ]:
def iterate_through_noise_values(df: pd.DataFrame, values: list[float]):
    results = [find_parents_then_group(df, ping_interval_noise_seconds) for ping_interval_noise_seconds in tqdm(values)]
    df_results = pd.DataFrame(
        results,
        columns=[
            "ping_interval_noise_secs",
            "num_parents",
            "num_trivial_dwell_secs",
            "perc_trivial_dwell_secs",
            "mean_dwell_secs",
            "median_dwell_secs",
            "num_users_under_60_secs",
            "perc_users_under_60_secs",
        ],
    )
    df_results.plot.scatter(
        x="ping_interval_noise_secs",
        y="num_trivial_dwell_secs",
        title="Trivial dwell time quantity vs. ping-interval noise seconds",
    )
    # df_results.plot.scatter(
    #     x="ping_interval_noise_secs", y="num_parents", title="Number of parent events vs. ping-interval noise seconds"
    # )
    plt.show()
    return df_results

In [ ]:
values = [
    0,
    0.01,
    0.02,
    0.05,
    0.1,
    0.2,
    0.3,
    0.5,
    0.8,
    1,
    2,
    5,
    10,
    20,
    30,
    60,
    100,
    200,
    300,
    600,
    1000,
    2000,  # 1800 seconds, or 30 minutes, is the maximum time between events in the same session since every session is at least 30 minutes apart
]
# values = [0, 0.01]

In [ ]:
results_afla = iterate_through_noise_values(df_afla, values)

In [ ]:
results_afla

In [ ]:
results_dfp = iterate_through_noise_values(df_dfp, values)

In [ ]:
results_dfp

In [ ]:
results_ov = iterate_through_noise_values(df_ov, values)

In [ ]:
results_ov

In [ ]:
results_19 = iterate_through_noise_values(df_19, values)

In [ ]:
results_19